# Проверка на своих картинках

Решает общий эксперт `all`, обученный на всех генераторах. Оценка картинки — sigmoid(логарифм шансов, что она
сгенерирована, минус порог): выше 0.5 значит сгенерирована. Порог пишет `scripts/run_test.py` в `thresholds.json`:
его превышает около 1 % реальных картинок валидации. Без этого файла порог 0, то есть обычная вероятность 0.5.

Для сгенерированной картинки называется семейство генератора: OpenAI, Google, Flux, Qwen и так далее.
Вероятности моделей одного семейства складываются.

На CPU одна картинка занимает около 0.2 секунды.

In [ ]:
import os, io, json, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import numpy as np
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from src.evaluate import groups
from src.model import Detector

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
size = cfg['train']['size']   # 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out, device, names=['all'])   # общий эксперт
family_of = {g: name for name, generators in cfg['model']['families'].items() for g in generators}
families = groups(model.experts['all'].models, family_of)   # семейство -> номера его моделей

# порог из теста, в логарифмах шансов; без него 0
threshold = 0.0
if (out / 'thresholds.json').exists():
    threshold = json.load(open(out / 'thresholds.json'))['threshold']
threshold

Картинка готовится как на тесте: не-JPEG один раз сжимается в JPEG с качеством 90 (на обучении так сжимались
все не-JPEG), потом картинка целиком сжимается до 512×512.

In [ ]:
def prepare(path):
    image = Image.open(path)
    fmt, image = image.format, image.convert('RGB')
    if fmt != 'JPEG':
        buf = io.BytesIO()
        image.save(buf, 'JPEG', quality=90)
        image = Image.open(buf).convert('RGB')
    image = image.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.array(image, dtype=np.float32) / 255).permute(2, 0, 1)   # [3, 512, 512], от 0 до 1


@torch.no_grad()
def predict(path):
    x = prepare(path)[None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        logits = model(x, ['all'])['all'][0].float()
    odds = (logits[1:].logsumexp(0) - logits[0]).item()   # логарифм шансов, что картинка сгенерирована
    share = logits[1:].softmax(0)                         # вероятности моделей, если она сгенерирована
    score = 1 / (1 + np.exp(threshold - odds))            # 0.5 = порог
    return score, {name: share[cols].sum().item() for name, cols in families.items()}

In [ ]:
path = 'my_images/example.jpg'   # своя картинка

score, result = predict(path)
family = max(result, key=result.get)
verdict = f'сгенерирована ({score:.2f}), семейство {family}' if score > 0.5 else f'реальная ({score:.2f})'
print(verdict)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(Image.open(path).convert('RGB'))
ax[0].set_title(verdict)
ax[0].axis('off')
order = sorted(result, key=result.get)
ax[1].barh(order, [result[n] for n in order], color='tab:red' if score > 0.5 else 'tab:gray')
ax[1].set_xlim(0, 1)
ax[1].set_title('семейство генератора, если картинка сгенерирована')
plt.tight_layout()
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')   # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        score, result = predict(p)
        family = max(result, key=result.get) if score > 0.5 else ''
        print(f"{'FAKE' if score > 0.5 else 'real'}  {score:.2f}  {family:12s}  {p.name}")